# **Global settings**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import rcParams
from matplotlib.pyplot import rc_context
import scanpy as sc
import scrublet as scr
import scvelo as scv
import seaborn as sns
import scipy.io
import os
import dotplot
import dotplot.utils
import math
import gseapy as gp
from gseapy.plot import barplot, dotplot, gseaplot
from gseapy.scipalette import SciPalette
from pylab import *
from matplotlib.colors import ListedColormap,LinearSegmentedColormap 

In [2]:
sc.settings.set_figure_params(dpi=100, dpi_save=300, figsize=(5, 5))

In [ ]:
plt.set_cmap('viridis')

In [4]:
scv.set_figure_params()

In [5]:
os.chdir('/disk213/xieqq/JINHUA138.sc')

# **CellLineage**

## **scrublet**

In [ ]:
output_file = 'Sample_1_SI_0_doublet.png'
input_dir = '/disk213/xieqq/sc/filtered_feature_bc_matrix/Sample_1_SI_0'
os.chdir(input_dir)

counts_matrix = scipy.io.mmread(input_dir + '/matrix.mtx').T.tocsc()
genes = np.array(scr.load_genes(input_dir + '/features.tsv', delimiter='\t', column=1))
out_df = pd.read_csv(input_dir + '/barcodes.tsv', header = None, index_col=None, names=['barcode'])

#print('Counts matrix shape: {} rows, {} columns'.format(counts_matrix.shape[0], counts_matrix.shape[1]))
#print('Number of genes in gene list: {}'.format(len(genes)))

In [ ]:
scrub = scr.Scrublet(counts_matrix, expected_doublet_rate=0.06)

In [ ]:
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, min_cells=3, min_gene_variability_pctl=85, n_prin_comps=30)

In [ ]:
scrub.call_doublets(threshold=0.25)

In [ ]:
scrub.plot_histogram()
plt.savefig(output_file,dpi=1000)

In [ ]:
out_df['doublet_scores'] = doublet_scores
out_df['predicted_doublets'] = predicted_doublets
out_df.to_csv(input_dir + '/doublet.txt', index=False,header=True)

## **standardized and concat**

In [ ]:
PATH='/disk213/xieqq/JINHUA138.sc/filtered_feature_bc_matrix/'
mtgene=pd.read_csv("/disk212/yupf/database/scRNA-seq/NewAtlas/mtgene.csv")
ap={}
sample=['Sample_1_SI_0','Sample_2_SI_0','Sample_3_SI_0','Sample_4_SI_60','Sample_5_SI_60',
        'Sample_6_SI_60','Sample_7_SI_90','Sample_8_SI_90','Sample_9_SI_90','Sample_10_SI_180',
        'Sample_11_SI_180','Sample_12_SI_180','Sample_13_SI_240','Sample_14_SI_240','Sample_15_SI_240',
        'Sample_16_LI_0','Sample_17_LI_0','Sample_18_LI_60','Sample_19_LI_60','Sample_20_LI_90',
        'Sample_21_LI_90','Sample_22_LI_180','Sample_23_LI_180','Sample_24_LI_240','Sample_25_LI_240']

for i in sample:
    ap[f'{i}']=sc.read_10x_mtx(PATH+f'{i}')
    ap[f'{i}'].var_names_make_unique
    scrublets=pd.read_csv(PATH+f'{i}'+'/doublet.txt',index_col='barcode')
    ap[f'{i}'].obs['doublet_scores']=scrublets['doublet_scores']
    ap[f'{i}'].obs['predicted_doublets']=scrublets['predicted_doublets']
    x=['{}',f'{i}']
    ap[f'{i}'].obs.index=ap[f'{i}'].obs.index.map('_'.join(x).format)
    
    sc.pp.filter_cells(ap[f'{i}'], min_genes=200)
    sc.pp.filter_genes(ap[f'{i}'], min_cells=3)
    # ap[f'{i}'].var['mt'] = ap[f'{i}'].var_names.str.startswith('MT')   #annotate the group of mitochondrial genes as 'mt'
    ap[f'{i}'].var['mt'] = ap[f'{i}'].var_names.isin(mtgene['MT-genes'])
    sc.pp.calculate_qc_metrics(ap[f'{i}'], qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    # sc.pl.violin(ap[f'{i}'],["n_genes_by_counts", "total_counts", "pct_counts_mt"],jitter=0.1,multi_panel=True,show=False,save='_'+i+'_QC.pdf')
    
    ap[f'{i}'] = ap[f'{i}'][ap[f'{i}'].obs.n_genes_by_counts < 7500, :]
    ap[f'{i}'] = ap[f'{i}'][ap[f'{i}'].obs.n_genes_by_counts > 200, :]
    ap[f'{i}'] = ap[f'{i}'][ap[f'{i}'].obs.pct_counts_mt < 50, :]

In [ ]:
sc.pl.violin(ap['Sample_25_LI_240'],["n_genes_by_counts", "total_counts", "pct_counts_mt"],jitter=0.1,multi_panel=True)

In [ ]:
adata=sc.concat(ap.values(),keys=ap.keys(),label='PRO1_JH')
adata

In [ ]:
sc.pl.violin(adata,["n_genes_by_counts", "total_counts", "pct_counts_mt"],jitter=False,stripplot=False, multi_panel=True,save='_afterQC.pdf')

In [142]:
adata.write('adata_rowcounts.h5ad')

In [145]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

adata.raw = adata
adata = adata[:, adata.var.highly_variable]
    
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)

adata = adata.raw.to_adata()

In [ ]:
mtgene=pd.read_csv("/disk212/yupf/database/scRNA-seq/NewAtlas/mtgene.csv")
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.isin(mtgene['MT-genes'])
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)
sc.pl.violin(adata,["n_genes_by_counts", "total_counts", "pct_counts_mt"],jitter=0.1,multi_panel=True)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)

In [ ]:
sc.external.pp.harmony_integrate(adata, 'BATCH', basis='X_pca', adjusted_basis='X_pca_harmony')

In [ ]:
sc.pp.neighbors(adata, use_rep="X_pca_harmony")

In [13]:
sc.tl.umap(adata, min_dist=0.1)

In [ ]:
sc.tl.tsne(adata,use_rep="X_pca_harmony")

In [15]:
sc.tl.leiden(adata)

In [ ]:
for i in range(1,10):
    sc.tl.leiden(adata,resolution=round(i*0.1,1),key_added=f'cluster_{round(i*0.1,1)}')
sc.pl.umap(adata,color=[f'cluster_{round(i*0.1,1)}'for i in range(1,10)],ncols=2,wspace=0.5)

In [18]:
adata.write('adata_concat.h5ad')

## **marker gene**

In [7]:
adata=sc.read_h5ad('adata_concat.h5ad')

In [ ]:
sc.tl.leiden(adata,resolution=0.4,key_added='cluster_0.4')
sc.pl.umap(adata,color=['cluster_0.4'],legend_loc='on data')

In [ ]:
GENE = ['EPCAM','KRT8','ELF3','SLC44A4']
sc.pl.umap(adata,color=GENE)  # epithelial cell genes
sc.pl.violin(adata, keys=GENE, groupby='cluster_0.4')

In [ ]:
GENE = ['JCHAIN','MZB1','DERL3']
sc.pl.umap(adata,color=GENE)  #plasma cell genes
sc.pl.violin(adata, keys=GENE, groupby='cluster_0.4')

In [ ]:
GENE = ['CD19','CD79A','CD79B','SMC6','PAX5','MEF2C','POU2AF1']
sc.pl.umap(adata,color=GENE)  #B cell genes
sc.pl.violin(adata, keys=GENE, groupby='cluster_0.4')

In [ ]:
GENE = ['CD3D','CD3E','CD4','CD8A','PRF1','NKG7','ZAP70','PRF1','CD2','CST7']
sc.pl.umap(adata,color=GENE) #T/ILC/NK cell genes
sc.pl.violin(adata, keys=GENE, groupby='cluster_0.4')

In [ ]:
GENE = ['STMN1','TUBB','STMN1','PTMA','RPS20','RPL37','RPS15A']
sc.pl.umap(adata,color=GENE) #neuronal cell
sc.pl.violin(adata, keys=GENE, groupby='cluster_0.4')

In [ ]:
GENE = ['PECAM1','CLDN5','HEY1','EFNB2','PROX1','APOA1','TIMP3','ICAM1']
sc.pl.umap(adata,color=GENE)  #endothelial cell
sc.pl.violin(adata, keys=GENE, groupby='cluster_0.4')

In [ ]:
GENE = ['ZEB2','BMP4','MMP3','CALD1']
sc.pl.umap(adata,color=GENE) #mesenchymal cell
sc.pl.violin(adata, keys=GENE, groupby='cluster_0.4')

In [ ]:
GENE = ['CD163','C1QB','C1QC','CD163','CD68','AIF1','TYROBP','FCER1A']
sc.pl.umap(adata,color=GENE)  #myeloid lineage
sc.pl.violin(adata, keys=GENE, groupby='cluster_0.4')

In [ ]:
def function(a):
    if a in ['0','1','5','7','8','11','12','16','17','18','19','20','21','22']:
        return 'Epithelial' 
    elif a in ['3','6']:
        return "Bcells"
    elif a in ['13']:
        return "Neuronal"
    elif a in ['9']:
        return "Plasma" 
    elif a in ['2','4']:
        return "T_ILC_NKcells"
    elif a in ['10','14']:
        return "Myeloid"
    elif a in ['15']:
        return "mass"
adata.obs["CellLineage"]=adata.obs.apply(lambda x: function(x['cluster_0.4']),axis=1) 

In [45]:
mass = adata[adata.obs['CellLineage'].isin(['mass'])]

In [ ]:
sc.tl.leiden(mass)
sc.pl.umap(mass,color=['leiden'],legend_loc='on data')

In [47]:
sc.tl.leiden(mass,resolution=0.1,key_added='cluster_0.1')

In [ ]:
sc.pl.umap(mass,color=['cluster_0.1'],legend_loc='on data')

In [ ]:
#endothelial cell
sc.pl.umap(mass,color=['PECAM1','CLDN5','HEY1','EFNB2','PROX1','APOA1','TIMP3','ICAM1'])

In [ ]:
#mesenchymal cell
sc.pl.umap(mass,color=['ZEB2','BMP4','MMP3','CALD1'])

In [ ]:
def function(a):
    if a in ['1']:
        return "Mesenchymal"
    elif a in ['0']:
        return "Endothelial"
df = mass.obs
df["CellLineage"] = df.apply(lambda x: function(x['cluster_0.1']),axis=1) 

In [50]:
adata.obs['CellLineage'] = adata.obs['CellLineage'].cat.add_categories('Endothelial')
adata.obs['CellLineage'] = adata.obs['CellLineage'].cat.add_categories('Mesenchymal')

In [51]:
adata.obs.loc[mass.obs_names,'CellLineage'] = mass.obs['CellLineage']

In [52]:
adata.obs['CellLineage'] = adata.obs['CellLineage'].cat.remove_unused_categories()

In [59]:
CellLineage_newcolors = ['#1b9e77','#7570b3','#EE9B00','#CA6702','#6699CC','#FF220C','#e7298a','#6F5E5C']

In [ ]:
sc.pl.umap(adata, color=['CellLineage'], palette=CellLineage_newcolors, save='_CellLineage.pdf')

In [62]:
adata.write('adata_CellLineage.h5ad')

# **CellType: Epithelial**

In [76]:
os.chdir('/disk213/xieqq/JINHUA138.sc')

In [120]:
adata=sc.read_h5ad('adata_CellLineage_rank_genes_groups.h5ad')

In [123]:
Epithelial=adata[adata.obs['CellLineage'].isin(['Epithelial'])]

In [ ]:
sc.tl.leiden(Epithelial)

In [ ]:
sc.tl.leiden(Epithelial,resolution=0.5,key_added='cluster_0.5')

## **marker gene**

In [ ]:
##Enterocytes
GENE = ['ANPEP','FABP2','CLCA4','SLC5A1','SI','ACE2']
sc.pl.umap(Epithelial,color=GENE)
sc.pl.violin(Epithelial, keys=GENE, groupby='cluster_0.5')
##Colonocytes
GENE = ['CA2','SLC26A2']
sc.pl.umap(Epithelial,color=GENE)
sc.pl.violin(Epithelial, keys=GENE, groupby='cluster_0.5')

In [ ]:
##BEST4 enterocytes
GENE = ['BEST4','GUCA2B','CFTR','NOTCH2']
sc.pl.umap(Epithelial,color=GENE)
sc.pl.violin(Epithelial, keys=GENE, groupby='cluster_0.5')

In [ ]:
##Goblet cells
GENE = ['CLCA1','SPDEF','TFF3','REG4','SPINK4','CXCL8']
sc.pl.umap(Epithelial,color=GENE)
sc.pl.violin(Epithelial, keys=GENE, groupby='cluster_0.5')

In [ ]:
##Tuft
GENE = ['POU2F3','IRAG2']
sc.pl.umap(Epithelial,color=GENE)
sc.pl.violin(Epithelial, keys=GENE, groupby='cluster_0.5')

In [ ]:
##EEC: Endometrial Epithelial Cells
GENE = ['CHGA','CHGB','NEUROD1']
sc.pl.umap(Epithelial,color=GENE)
sc.pl.violin(Epithelial, keys=GENE, groupby='cluster_0.5')

In [ ]:
##Stem cells
sc.pl.umap(Epithelial,color=['LGR5','SMOC2','RGMB','AXIN2'])
##TA: transit-amplifying
sc.pl.umap(Epithelial,color=['TOP2A','PCNA'])
##Progenitor cell
sc.pl.umap(Epithelial,color=['CDK6','AKAP7','RBPJ'])

##intestinal revival stem cells
#sc.pl.umap(Epithelial,color=['CLU'])
##proximal progenitors
#sc.pl.umap(Epithelial,color=['FGG','BEX5'])
##distal progenitors
#sc.pl.umap(Epithelial,color=['CKB','AKAP7'])
##pancreatic progenitors
#sc.pl.umap(Epithelial,color=['RBPJ','CPA1'])

In [ ]:
##Microfold
##sc.pl.umap(Epithelial,color=['SPIB','CCL20','GP2'])

##Paneth cells
# sc.pl.umap(Epithelial,color=['CA4','PIGR'])

In [ ]:
def function(a):
    if a in ['0','2','4','13','14','15','16','20']:
        return "Enterocytes"
    elif a in ['6','7','10','18']:
        return "Colonocytes"
    elif a in ['8','12']:
        return "BEST4enterocytes"
    elif a in ['9']:
        return "Tuft"
    elif a in ['3']:
        return "Goblet"
    elif a in ['11']:
        return "EECs"
    elif a in ['17','19']:
        return "TA"
    elif a in ['1']:
        return "mass1"
    elif a in ['5']:
        return "mass2"
df = Epithelial.obs
df["CellType"] = df.apply(lambda x: function(x['cluster_0.5']),axis=1) 

In [138]:
mass1 = Epithelial[Epithelial.obs['CellType'].isin(['mass1'])]

In [ ]:
sc.tl.leiden(mass1)
sc.tl.leiden(mass1,resolution=0.5,key_added='mass_cluster_0.5')
sc.pl.umap(mass1,color=['leiden','mass_cluster_0.5'],legend_loc='on data',legend_fontsize='xx-small')
sc.pl.umap(mass1,color=['leiden','mass_cluster_0.5'])

In [ ]:
##Enterocytes
GENE = ['ANPEP','FABP2','CLCA4','SLC5A1','SI','ACE2']
sc.pl.umap(mass1,color=GENE)
sc.pl.violin(mass1, keys=GENE, groupby='leiden')
##Colonocytes
GENE = ['CA2','SLC26A2']
sc.pl.umap(mass1,color=GENE)
sc.pl.violin(mass1, keys=GENE, groupby='leiden')

In [ ]:
#Stem cells
GENE = ['LGR5','SMOC2','RGMB','AXIN2']
sc.pl.umap(mass1,color=GENE)
sc.pl.violin(mass1, keys=GENE, groupby='leiden')
##TA: transit-amplifying
GENE = ['TOP2A','PCNA']
sc.pl.umap(mass1,color=GENE)
sc.pl.violin(mass1, keys=GENE, groupby='leiden')
#Progenitor cell
GENE = ['CDK6','AKAP7','RBPJ']
sc.pl.umap(mass1,color=GENE)
sc.pl.violin(mass1, keys=GENE, groupby='leiden')

In [ ]:
def function(a):
    if a in ['4']:
        return "TA"
    elif a in ['0','1','2','5','6','8','9','11','12','13','14','15']:
        return "Enterocytes"
    elif a in ['3','7','10']:
        return "Colonocytes"
df = mass1.obs
df["CellType"] = df.apply(lambda x: function(x['leiden']),axis=1) 

In [231]:
Epithelial.obs.loc[mass1.obs_names,'CellType'] = mass1.obs['CellType'].astype('object')

In [232]:
Epithelial.obs['CellType'] = Epithelial.obs['CellType'].cat.remove_unused_categories()

In [ ]:
TA = Epithelial[Epithelial.obs['CellType'].isin(['TA'])]
sc.tl.leiden(TA)
sc.tl.leiden(TA,resolution=0.5,key_added='mass_cluster_0.5')
sc.pl.umap(TA,color=['leiden','mass_cluster_0.5'],legend_loc='on data',legend_fontsize='xx-small')
sc.pl.umap(TA,color=['leiden','mass_cluster_0.5'])

In [ ]:
##Enterocytes
GENE = ['ANPEP','FABP2','CLCA4','SLC5A1','SI','ACE2']
sc.pl.umap(TA,color=GENE)
sc.pl.violin(TA, keys=GENE, groupby='mass_cluster_0.5')
##Colonocytes
GENE = ['CA2','SLC26A2']
sc.pl.umap(TA,color=GENE)
sc.pl.violin(TA, keys=GENE, groupby='mass_cluster_0.5')

In [ ]:
#Stem cells
# GENE = ['LGR5','SMOC2','RGMB','AXIN2']
# sc.pl.umap(TA,color=GENE)
# sc.pl.violin(TA, keys=GENE, groupby='mass_cluster_0.5')
##TA: transit-amplifying
GENE = ['TOP2A','PCNA']
sc.pl.umap(TA,color=GENE)
sc.pl.violin(TA, keys=GENE, groupby='mass_cluster_0.5')
#Progenitor cell
GENE = ['CDK6','AKAP7','RBPJ']
sc.pl.umap(TA,color=GENE)
sc.pl.violin(TA, keys=GENE, groupby='mass_cluster_0.5')

In [ ]:
def function(a):
    if a in ['0','2','4','6']:
        return "TA"
    elif a in ['1','5','7']:
        return "Progenitor"
    elif a in ['3']:
        return "Enterocytes"
df = TA.obs
df["CellType"] = df.apply(lambda x: function(x['mass_cluster_0.5']),axis=1) 

In [ ]:
Epithelial.obs['CellType'].cat.categories.tolist()

In [ ]:
TA.obs['CellType'].cat.categories.tolist()

In [234]:
Epithelial.obs['CellType'] = Epithelial.obs['CellType'].cat.add_categories('Progenitor')

In [235]:
Epithelial.obs.loc[TA.obs_names,'CellType'] = TA.obs['CellType'].astype('object')

In [155]:
mass2 = Epithelial[Epithelial.obs['CellType'].isin(['mass2'])]

In [ ]:
sc.tl.leiden(mass2)
sc.tl.leiden(mass2,resolution=0.4,key_added='mass_cluster_0.4')
sc.pl.umap(mass2,color=['leiden','mass_cluster_0.4'],legend_loc='on data',legend_fontsize='xx-small')
sc.pl.umap(mass2,color=['leiden','mass_cluster_0.4'])

In [ ]:
##Enterocytes
GENE = ['ANPEP','FABP2','CLCA4','SLC5A1','SI','ACE2']
sc.pl.umap(mass2,color=GENE)
sc.pl.violin(mass2, keys=GENE, groupby='leiden')
##Colonocytes
GENE = ['CA2','SLC26A2']
sc.pl.umap(mass2,color=GENE)
sc.pl.violin(mass2, keys=GENE, groupby='leiden')

In [ ]:
#Stem cells
GENE = ['LGR5','SMOC2','RGMB','AXIN2']
sc.pl.umap(mass2,color=GENE)
sc.pl.violin(mass2, keys=GENE, groupby='leiden')
##TA: transit-amplifying
GENE = ['TOP2A','PCNA']
sc.pl.umap(mass2,color=GENE)
sc.pl.violin(mass2, keys=GENE, groupby='leiden')
#Progenitor cell
GENE = ['CDK6','AKAP7','RBPJ']
sc.pl.umap(mass2,color=GENE)
sc.pl.violin(mass2, keys=GENE, groupby='leiden')

In [ ]:
def function(a):
    if a in ['1','11']:
        return "Stem"
    elif a in ['6','10','12']:
        return "Progenitor"
    elif a in ['2','4','8','9']:
        return "TA"
    elif a in ['3','5','7','13','14']:
        return "Enterocytes"
    elif a in ['0']:
        return "Colonocytes"
df = mass2.obs
df["CellType"] = df.apply(lambda x: function(x['leiden']),axis=1) 

In [238]:
Epithelial.obs['CellType'] = Epithelial.obs['CellType'].cat.add_categories('Stem')

In [239]:
Epithelial.obs.loc[mass2.obs_names,'CellType'] = mass2.obs['CellType'].astype('object')

In [240]:
Epithelial.obs['CellType'] = Epithelial.obs['CellType'].cat.remove_unused_categories()

In [21]:
Epithelial_newcolors = ['#004B23','#007200','#38B000','#52B788','#95D5B2','#D8F3DC','#34A0A4','#1A759F','#184E77']

In [ ]:
sc.pl.umap(Epithelial, color=['CellType'], palette=Epithelial_newcolors, save='_Epithelial.pdf')

In [253]:
Epithelial.write_h5ad('Epithelial_CellType.h5ad')

# **Public Data**

In [8]:
os.chdir('/disk213/xieqq/JINHUA138.sc/Public_data')

## **Human**

### **Data**

In [ ]:
adata=sc.read_h5ad('/disk213/xieqq/JINHUA138.sc/Public_data/data/Full_obj_log_counts_soupx_v2.h5ad')   #PRJNA666217

In [ ]:
sc.pl.violin(adata,["n_genes_by_counts", "pct_counts_mt"],jitter=False,stripplot=False, multi_panel=True, save='_Human_beforeQC.pdf')

In [ ]:
adata = adata[adata.obs.n_genes_by_counts < 7500, :]
adata = adata[adata.obs.n_genes_by_counts > 200, :]
adata = adata[adata.obs.pct_counts_mt < 50, :]
sc.pl.violin(adata,["n_genes_by_counts", "pct_counts_mt"],jitter=False,stripplot=False, multi_panel=True, save='_Human_afterQC.pdf')

In [ ]:
def function(a):
    if a in ['DUO','JEJ','FMIL','FPIL','FTIL','ILE','ILE1','ILE2','TIL']:
        return 'small' 
    elif a in ['CAE','ACL','DCL','TCL','SCL']:
        return "large" 
adata.obs["INTESTINAL"]=adata.obs.apply(lambda x: function(x['Region code']),axis=1)
adata.obs["INTESTINAL"]=adata.obs["INTESTINAL"].astype('category')

In [14]:
adata.obs["TIME"]=adata.obs["Age"]
adata.obs["TIME"].replace({"4": "4y", "6": "6y", "9": "9y", "10": "10y", "11": "11y", "12": "12y", "13": "13y", "14": "14y",
                           "20-25": "20-25y", "25-30": "25-30y", "45-50": "45-50y", "60-65": "60-65y", "65-70": "65-70y", "70-75": "70-75y"}, inplace=True)
adata.obs["TIME"]=adata.obs["TIME"].astype('category')

In [ ]:
adata = adata[adata.obs['INTESTINAL'].isin(['small','large'])]
adata = adata[adata.obs['Diagnosis'].isin(['Healthy adult','fetal','Pediatric healthy'])]

In [466]:
# adata.obs["CellLineage"]=adata.obs["category"]
# adata.obs["CellLineage"]=adata.obs["CellLineage"].astype('category')
# adata.obs["CellType"]=adata.obs["Integrated_05"]
# adata.obs["CellType"]=adata.obs["CellType"].astype('category')

In [ ]:
# adata.obs['Sample name'].cat.categories.tolist()
# adata.obs['Diagnosis'].cat.categories.tolist()
# adata.obs['Age'].cat.categories.tolist()
# adata.obs['sample name'].cat.categories.tolist()
# adata.obs['Region code'].cat.categories.tolist()
# adata.obs['Fraction'].cat.categories.tolist()
# adata.obs['Gender'].cat.categories.tolist()
# adata.obs['Region'].cat.categories.tolist()
# adata.obs['batch'].cat.categories.tolist()
# adata.obs['category'].cat.categories.tolist()
# adata.obs['Age_group'].cat.categories.tolist()
# adata.obs['Integrated_05'].cat.categories.tolist()

### **PCA+harmony+reduction+cluster**

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)

In [ ]:
sc.external.pp.harmony_integrate(adata, 'batch', basis='X_pca', adjusted_basis='X_pca_harmony')

In [ ]:
sc.pp.neighbors(adata, use_rep="X_pca_harmony")

In [23]:
sc.tl.umap(adata, min_dist=0.1)

In [24]:
sc.tl.tsne(adata,use_rep="X_pca_harmony")

### **annotation**

In [ ]:
sc.tl.leiden(adata)
sc.pl.umap(adata,color=['leiden'],legend_loc='on data')

In [ ]:
GENE = ['EPCAM','KRT8','ELF3','SLC44A4']
sc.pl.umap(adata,color=GENE)  # epithelial cell genes
with rc_context({'figure.figsize':(15,5)}): 
    sc.pl.violin(adata, keys=GENE, groupby='leiden')

In [ ]:
def function(a):
    if a in ['0','1','2','5','11','16','20','24','28','29','32','36','39','40','41']:
        return 'Epithelial' 
adata.obs["CellLineage"]=adata.obs.apply(lambda x: function(x['leiden']),axis=1) 

In [28]:
adata.obs["CellType"]=adata.obs["CellLineage"]

In [29]:
Epithelial=adata[adata.obs['CellLineage'].isin(['Epithelial'])]

In [ ]:
sc.tl.leiden(Epithelial)
sc.pl.umap(Epithelial,color=['leiden'],legend_loc='on data')

In [ ]:
##Enterocytes
GENE = ['ANPEP','FABP2','CLCA4','SLC5A1','SI','ACE2']
sc.pl.umap(Epithelial,color=GENE)
with rc_context({'figure.figsize':(15,5)}): 
    sc.pl.violin(Epithelial, keys=GENE, groupby='leiden')
##Colonocytes
GENE = ['CA2','SLC26A2']
sc.pl.umap(Epithelial,color=GENE)
with rc_context({'figure.figsize':(15,5)}): 
    sc.pl.violin(Epithelial, keys=GENE, groupby='leiden')

In [ ]:
def function(a):
    if a in ['0','2','4','13','14','15','16','20']:
        return "Enterocytes"
    elif a in ['6','7','10','18']:
        return "Colonocytes"
df = Epithelial.obs
df["CellType"] = df.apply(lambda x: function(x['leiden']),axis=1) 

In [40]:
adata.obs['CellType'] = adata.obs['CellType'].cat.add_categories('Enterocytes')
adata.obs['CellType'] = adata.obs['CellType'].cat.add_categories('Colonocytes')
adata.obs.loc[Epithelial.obs_names,'CellType'] = Epithelial.obs['CellType'].astype('object')
adata.obs['CellType'] = adata.obs['CellType'].cat.remove_unused_categories()

In [42]:
adata.write('Human_intestine.h5ad')

## **Commercial pig**

### **Data**

In [ ]:
adata=sc.read_h5ad('/disk213/xieqq/JINHUA138.sc/Public_data/data/atlas_level2.h5ad')

In [ ]:
sc.pl.violin(adata,["n_genes_by_counts", "pct_counts_mt"],jitter=False,stripplot=False, multi_panel=True, save='_CP_beforeQC.pdf')

In [ ]:
adata = adata[adata.obs.n_genes_by_counts < 7500, :]
adata = adata[adata.obs.n_genes_by_counts > 200, :]
adata = adata[adata.obs.pct_counts_mt < 50, :]
sc.pl.violin(adata,["n_genes_by_counts", "pct_counts_mt"],jitter=False,stripplot=False, multi_panel=True, save='_CP_afterQC.pdf')

In [48]:
data = pd.read_csv('/disk213/xieqq/JINHUA138.sc/Public_data/data/GEO_metadata.csv')

In [ ]:
mapping_dict_breed = dict(zip(data['Dataset'], data['breed']))
mapping_dict_stage = dict(zip(data['Dataset'], data['stage']))

In [ ]:
adata.obs['BREED'] = adata.obs['project'].map(mapping_dict_breed)
adata.obs['TIME'] = adata.obs['project'].map(mapping_dict_stage)
adata.obs["BREED"]=adata.obs["BREED"].astype('category')
adata.obs["TIME"]=adata.obs["TIME"].astype('category')

In [ ]:
adata = adata[adata.obs['INTESTINAL'].isin(['small','large'])]
adata = adata[adata.obs['BREED'].isin(['Duroc','LY','large white','DLY','Mixed'])]
adata = adata[adata.obs['TIME'].isin(['0d','1d','3d','5d','7d','14d','21d','49d','60d','90d','160d','180d','240d'])]

### **PCA+harmony+reduction+cluster**

In [56]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)

In [ ]:
sc.external.pp.harmony_integrate(adata, 'project', basis='X_pca', adjusted_basis='X_pca_harmony')

In [ ]:
sc.pp.neighbors(adata, use_rep="X_pca_harmony")

In [62]:
sc.tl.umap(adata, min_dist=0.1)

In [63]:
sc.tl.tsne(adata,use_rep="X_pca_harmony")

In [65]:
adata.write('CP_intestine.h5ad')

### **annotation**

In [ ]:
adata=sc.read_h5ad('CP_intestine.h5ad')

In [ ]:
sc.tl.leiden(adata)
sc.pl.umap(adata,color=['leiden'],legend_loc='on data')

In [ ]:
GENE = ['EPCAM','KRT8','ELF3','SLC44A4']
sc.pl.umap(adata,color=GENE)  # epithelial cell genes
with rc_context({'figure.figsize':(15,5)}): 
    sc.pl.violin(adata, keys=GENE, groupby='leiden')

In [ ]:
def function(a):
    if a in ['4','7','9','14','15','19','20','21','22','24','25','26','29','30','33','34']:
        return 'Epithelial' 
adata.obs["CellLineage"]=adata.obs.apply(lambda x: function(x['leiden']),axis=1) 

In [77]:
adata.obs["CellType"]=adata.obs["CellLineage"]

In [79]:
Epithelial=adata[adata.obs['CellLineage'].isin(['Epithelial'])]

In [ ]:
sc.tl.leiden(Epithelial)
sc.pl.umap(Epithelial,color=['leiden'],legend_loc='on data',legend_fontsize='xx-small')

In [ ]:
##Enterocytes
GENE = ['ANPEP','FABP2','CLCA4','SLC5A1','SI','ACE2']
sc.pl.umap(Epithelial,color=GENE)
with rc_context({'figure.figsize':(15,5)}): 
    sc.pl.violin(Epithelial, keys=GENE, groupby='leiden')
##Colonocytes
GENE = ['CA2','SLC26A2']
sc.pl.umap(Epithelial,color=GENE)
with rc_context({'figure.figsize':(15,5)}): 
    sc.pl.violin(Epithelial, keys=GENE, groupby='leiden')

In [ ]:
def function(a):
    if a in ['2','9','11','13','17','18','24','26','27','28','33','34','38']:
        return "Enterocytes"
    elif a in ['3','8','14','15','31','35']:
        return "Colonocytes"
df = Epithelial.obs
df["CellType"] = df.apply(lambda x: function(x['leiden']),axis=1) 

In [87]:
adata.obs['CellType'] = adata.obs['CellType'].cat.add_categories('Enterocytes')
adata.obs['CellType'] = adata.obs['CellType'].cat.add_categories('Colonocytes')
adata.obs.loc[Epithelial.obs_names,'CellType'] = Epithelial.obs['CellType'].astype('object')
adata.obs['CellType'] = adata.obs['CellType'].cat.remove_unused_categories()

In [90]:
adata.write('CP_intestine.h5ad')

## **Merge (shared genes)**

In [5]:
os.chdir('/disk213/xieqq/JINHUA138.sc/Public_data')

In [ ]:
adata_Jinhua=sc.read_h5ad('/disk213/xieqq/JINHUA138.sc/Epithelial_CellType.h5ad')
var_names_df = pd.DataFrame(adata_Jinhua.var_names, columns=['gene_Jinhua'])
var_names_df.to_csv('gene_Jinhua.csv', index=False)

adata_CP=sc.read_h5ad('CP_intestine.h5ad')
var_names_df = pd.DataFrame(adata_CP.var_names, columns=['gene_CP'])
var_names_df.to_csv('gene_CP.csv', index=False)

adata_Human=sc.read_h5ad('Human_intestine.h5ad')
var_names_df = pd.DataFrame(adata_Human.var_names, columns=['gene_Human'])
var_names_df.to_csv('gene_Human.csv', index=False)

gene_intersect = pd.read_csv('gene_intersect.csv', header=1, names=['intersect'])

In [ ]:
replace_df = pd.read_csv('gene_Jinhua.csv', header=1, names=['gene_Jinhua', 'To'])

for index, row in replace_df.iterrows():
    gene = str(row['gene_Jinhua'])
    replace_gene = str(row['To'])
    pattern = r'\b{}\b'.format(re.escape(gene)) 
    adata_Jinhua.var_names = adata_Jinhua.var_names.str.replace(pattern, replace_gene, regex=True)

In [226]:
adata_Jinhua=adata_Jinhua[:, ~adata_Jinhua.var_names.isin(['undefined'])]
adata_Jinhua.var_names_make_unique()

In [186]:
replace_df = pd.read_csv('gene_CP.csv', header=1, names=['gene_CP', 'To'])

for index, row in replace_df.iterrows():
    gene = str(row['gene_CP'])
    replace_gene = str(row['To'])
    pattern = r'\b{}\b'.format(re.escape(gene))
    adata_CP.var_names = adata_CP.var_names.str.replace(pattern, replace_gene, regex=True)

In [192]:
adata_CP=adata_CP[:, ~adata_CP.var_names.isin(['undefined'])]
adata_CP.var_names_make_unique()

In [188]:
replace_df = pd.read_csv('gene_Human.csv', header=1, names=['gene_Human', 'To'])

for index, row in replace_df.iterrows():
    gene = str(row['gene_Human'])
    replace_gene = str(row['To'])
    pattern = r'\b{}\b'.format(re.escape(gene))
    adata_Human.var_names = adata_Human.var_names.str.replace(pattern, replace_gene, regex=True)

In [191]:
adata_Human=adata_Human[:, ~adata_Human.var_names.isin(['undefined'])]
adata_Human.var_names_make_unique()

In [227]:
adata_Human.obs["PROJECT"]=adata_Human.obs["batch"]
adata_Human.obs["SPECIES"]="Human"
adata_Human.obs["BREED"]="Human"
adata_Human = adata_Human[adata_Human.obs['TIME'].isin(['4y','6y','9y','10y','12y','20-25y','25-30y','45-50y','60-65y','65-70y','70-75y'])]

adata_Jinhua.obs["PROJECT"]=adata_Jinhua.obs["BATCH"]
adata_Jinhua.obs["SPECIES"]="Pig"
adata_Jinhua.obs["BREED"]="Jinhua"
# adata_Jinhua.obs['TIME'] = adata_Jinhua.obs['TIME'].astype(str) + 'd'
# adata_Jinhua.obs["TIME"]=adata_Jinhua.obs["TIME"].astype('category')

adata_CP.obs["PROJECT"]=adata_CP.obs["project"]
adata_CP.obs["SPECIES"]="Pig"
#adata_CP.obs["BREED"]=adata_CP.obs["BREED"]

In [ ]:
adata = sc.concat([adata_Jinhua, adata_CP, adata_Human], label='CONCAT')
adata = adata[adata.obs['CellType'].isin(['Enterocytes', 'Colonocytes'])]

In [231]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)  
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

In [232]:
sc.tl.pca(adata, svd_solver='arpack')
sc.external.pp.harmony_integrate(adata, 'PROJECT', basis='X_pca', adjusted_basis='X_pca_harmony')
sc.pp.neighbors(adata, use_rep="X_pca_harmony")
sc.tl.umap(adata, min_dist=0.1)
sc.tl.tsne(adata,use_rep="X_pca_harmony")

In [234]:
adata.write('Concat_intestine.h5ad')

In [ ]:
print(adata.obs['CellType'].cat.categories.tolist())
print(adata.obs['TIME'].cat.categories.tolist())
print(adata.obs['BREED'].cat.categories.tolist())
print(adata.obs['SPECIES'].cat.categories.tolist())
print(adata.obs['PROJECT'].cat.categories.tolist())

In [ ]:
sc.pl.umap(adata, color=['PROJECT'], save='_PROJECT.pdf')
# sc.pl.tsne(adata, color=['PROJECT'], save='_PROJECT.pdf')

In [243]:
adata.obs['SOURCE']=adata.obs['PROJECT']
our_study=['ce0','ce60','ce90','ce180','ce240','co0','co60','co90','co180','co240','du0','du60','du90','du180','du240','il0','il60','il90','il180','il240','je0','je60','je90','je180','je240','DUR_DU','DUR_JE','DUR_IL','DUR_CE','DUR_CO','EB5_DU','EB5_JE','EB5_IL','EB5_CE','EB5_CO']
pro_PRJNA681248=['PRJNA681248_ileum0','PRJNA681248_ileum1','PRJNA681248_ileum3','PRJNA681248_ileum7','PRJNA681248_ileum14','PRJNA681248_ileum21']
pro_PRJNA728441=['PRJNA728441_NSP1','PRJNA728441_NSP2','PRJNA728441_NSP3','PRJNA728441_PWP1','PRJNA728441_PWP2','PRJNA728441_PWP3']
pro_PRJNA685448=['PRJNA685448_Piglet_day0','PRJNA685448_Piglet_day7','PRJNA685448_Piglet_day14','PRJNA685448_Piglet_day21',]
pro_PRJNA859792=['PRJNA859792_DUOD1','PRJNA859792_DUOD2','PRJNA859792_DUOD3','PRJNA859792_DUOD4','PRJNA859792_JEJ1','PRJNA859792_JEJ2','PRJNA859792_JEJ3','PRJNA859792_JEJ4','PRJNA859792_IPP1','PRJNA859792_IPP2','PRJNA859792_IPP3','PRJNA859792_IPP4','PRJNA859792_NoPP1','PRJNA859792_NoPP2','PRJNA859792_NoPP4']
pro_gutcellatlas=['4918STDY7333456','4918STDY7702680','4918STDY7844899','4918STDY7447825','4918STDY7923744','4918STDY7389431','4918STDY7274839','4918STDY7844898','4918STDY7702679','4918STDY7714149','4918STDY7714150','Human_colon_16S8000471',
                  'Human_colon_16S8000473','Human_colon_16S8000475','Human_colon_16S8000477','Human_colon_16S8000479','Human_colon_16S8000487','Human_colon_16S8000489','Human_colon_16S8000491','Human_colon_16S8000493','Human_colon_16S8000511','Human_colon_16S8000513','Human_colon_16S8000515','Human_colon_16S8001863','Human_colon_16S8001865','Human_colon_16S8001867','Human_colon_16S8001869','Human_colon_16S8001871','Human_colon_16S8001878','Human_colon_16S8001879','Human_colon_16S8001881','Human_colon_16S8001883','Human_colon_16S8001885','Human_colon_16S8001903','Human_colon_16S8001905','Human_colon_16S8001907','Human_colon_16S8002566','Human_colon_16S8002581','Human_colon_16S8002582','Human_colon_16S8002623','Human_colon_16S8002624','Human_colon_16S8002626','Human_colon_16S8002627','Human_colon_16S8002628','Human_colon_16S8002629','Human_colon_16S8002630','Human_colon_16S8117829','Human_colon_16S8117830','Human_colon_16S8117831','Human_colon_16S8123908','Human_colon_16S8123910','Human_colon_16S8123911','Human_colon_16S8123912','Human_colon_16S8123913','Human_colon_16S8123915','Human_colon_16S8123916','Human_colon_16S8123917','Human_colon_16S8123918','Human_colon_16S8123920','Human_colon_16S8159191','Human_colon_16S8159193',
                  'WTDAtest7770716','WTDAtest7770717','WTDAtest7770718','WTDAtest7770719','WTDAtest7844017','WTDAtest7844018','WTDAtest7844019','WTDAtest7844020','WTDAtest7844022','WTDAtest7844024','WTDAtest7844026','WTDAtest7844027','WTDAtest7844029']

sources=['our_study','pro_PRJNA681248','pro_PRJNA728441','pro_PRJNA685448','pro_PRJNA859792','pro_gutcellatlas']
for source in sources:
    adata.obs['SOURCE']=adata.obs['SOURCE'].cat.add_categories(source)

adata.obs.loc[adata.obs['PROJECT'].isin(our_study),'SOURCE']='our_study'
adata.obs.loc[adata.obs['PROJECT'].isin(pro_PRJNA681248),'SOURCE']='pro_PRJNA681248'
adata.obs.loc[adata.obs['PROJECT'].isin(pro_PRJNA728441),'SOURCE']='pro_PRJNA728441'
adata.obs.loc[adata.obs['PROJECT'].isin(pro_PRJNA685448),'SOURCE']='pro_PRJNA685448'
adata.obs.loc[adata.obs['PROJECT'].isin(pro_PRJNA859792),'SOURCE']='pro_PRJNA859792'
adata.obs.loc[adata.obs['PROJECT'].isin(pro_gutcellatlas),'SOURCE']='pro_gutcellatlas'#Human
adata.obs['SOURCE']=adata.obs['SOURCE'].cat.remove_unused_categories()

In [ ]:
adata.obs['SOURCE'].unique()

In [ ]:
labels = ['our_study', 'pro_PRJNA681248', 'pro_PRJNA728441', 'pro_PRJNA685448', 'pro_PRJNA859792', 'pro_gutcellatlas']
adata.obs['Source_change_categories']=pd.Categorical(adata.obs['SOURCE'],categories=labels)
Source_newcolors = ['#FFADAD','#FFD6A5','#FFD6A5','#CAFFBF','#9BF6FF','#A0C4FF']
sc.pl.umap(adata, color=['Source_change_categories'], palette=Source_newcolors, save='_Source.pdf')
# sc.pl.tsne(adata, color=['Source_change_categories'], palette=Source_newcolors, save='_Source.pdf')

In [ ]:
labels = ['Enterocytes', 'Colonocytes']
adata.obs['CellType_change_categories']=pd.Categorical(adata.obs['CellType'],categories=labels)
CellType_newcolors = ['#004B23','#52B788']
sc.pl.umap(adata, color=['CellType_change_categories'], palette=CellType_newcolors, save='_CellType.pdf')
# sc.pl.tsne(adata, color=['CellType_change_categories'], palette=CellType_newcolors, save='_CellType.pdf')

In [ ]:
labels = ['0','60','90','180','240', #JH
          '0d','1d','3d','7d','14d','21d','49d','160d',  #CP
          '4y','6y','9y','10y','12y','20-25y','25-30y','45-50y','60-65y','65-70y','70-75y']   #Human
adata.obs['Time_change_categories']=pd.Categorical(adata.obs['TIME'],categories=labels)
Time_newcolors = ['#FF595E','#FFCA3A','#8AC926','#1982C4','#6A4C93',
                  '#FAE0E4','#F9BEC7','#FF99AC','#FF7096','#FF477E','#FF0A54','#FFEA00','#FFA200',
                  '#74C69D','#40916C','#2D6A4F','#1B4332','#081C15','#48CAE4','#0096C7','#023E8A','#C77DFF','#9D4EDD','#5A189A']
sc.pl.umap(adata, color=['Time_change_categories'], palette=Time_newcolors, save='_Time.pdf')
# sc.pl.tsne(adata, color=['Time_change_categories'], palette=Time_newcolors, save='_Time.pdf')

In [ ]:
labels = ['Human','Jinhua','Duroc','LY','large white','Mixed']   #Human
adata.obs['Breed_change_categories']=pd.Categorical(adata.obs['BREED'],categories=labels)
Breed_newcolors = ['#F94144','#F3722C','#F9C74F','#90BE6D','#43AA8B','#577590']
sc.pl.umap(adata, color=['Breed_change_categories'], palette=Breed_newcolors, save='_Breed.pdf')
# sc.pl.tsne(adata, color=['Breed_change_categories'], palette=Breed_newcolors, save='_Breed.pdf')

In [ ]:
labels = ['Human','Pig']   #Human
adata.obs['Species_change_categories']=pd.Categorical(adata.obs['SPECIES'],categories=labels)
Species_newcolors = ["#B499C9","#D95B5B"]
sc.pl.umap(adata, color=['Species_change_categories'], palette=Species_newcolors, save='_Species.pdf')
# sc.pl.tsne(adata, color=['Species_change_categories'], palette=Species_newcolors, save='_Species.pdf')

In [254]:
def function(a):
    if a in ['Jinhua']:
        return 'JinhuaPig' 
    elif a in ['large white', 'LY', 'Duroc', 'Mixed']:
        return "CommercialPig"
    elif a in ['Human']:
        return "Human"
adata.obs["BREEDadd"]=adata.obs.apply(lambda x: function(x['BREED']),axis=1)
adata.obs["BREEDadd"]=adata.obs["BREEDadd"].astype('category')

In [269]:
adata.obs["TIMEadd"]=adata.obs["TIME"]
adata.obs['TIMEadd'] = adata.obs['TIMEadd'].cat.add_categories('60d')
adata.obs['TIMEadd'] = adata.obs['TIMEadd'].cat.add_categories('90d')
adata.obs['TIMEadd'] = adata.obs['TIMEadd'].cat.add_categories('180d')
adata.obs['TIMEadd'] = adata.obs['TIMEadd'].cat.add_categories('240d')

mask = adata.obs['BREED'] == 'Jinhua'
adata.obs.loc[mask, 'TIMEadd'] = adata.obs.loc[mask, 'TIMEadd'].astype(str) + 'd'
adata.obs["TIMEadd"]=adata.obs["TIMEadd"].astype('category')

adata.obs['TIMEadd'] = adata.obs['TIMEadd'].cat.remove_unused_categories()

In [277]:
adata.write('Concat_intestine.h5ad')

## Pseudobulk RNA-seq

In [ ]:
#source /disk211/public/anaconda3/bin/activate /disk211/public/anaconda3/envs/labBase
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import decoupler as dc
import scanpy as sc
import os
import math
import csv

In [ ]:
adata=sc.read_h5ad('/disk213/xieqq/JINHUA138.sc/Public_data/Concat_intestine.h5ad')

In [ ]:
adata.layers['counts']=adata.X
pdata = dc.get_pseudobulk(
    adata,
    sample_col='BREEDadd-INTESTINAL-TIME',
    groups_col=None,
    layer='counts',
    mode='sum',
    min_cells=10,
    min_counts=1000
)
pdata.layers['counts'] = pdata.X.copy()

# Normalize, scale and compute pca
sc.pp.normalize_total(pdata, target_sum=1e4)
sc.pp.log1p(pdata)
sc.pp.scale(pdata, max_value=10)
sc.tl.pca(pdata)
list = pdata.T.to_df()
list.to_csv("pseudobulk_raw_counts_scaled.csv", index=True)

In [ ]:
Glist = adata.var_names
new_data = pd.DataFrame(columns=adata.obs['SPECIES-INTESTINAL-TIMEadd'].unique())
for gene_name in Glist[:]:
    expression = adata[:, gene_name].X
    mean_expression_by_cluster = []
    list = []
    for cluster in adata.obs['SPECIES-INTESTINAL-TIMEadd'].unique():
        cells_in_cluster = adata.obs['SPECIES-INTESTINAL-TIMEadd'] == cluster
        mean_expression = expression[cells_in_cluster].mean()
        mean_expression_by_cluster = mean_expression_by_cluster + [mean_expression]
    new_data = new_data.append(pd.Series(mean_expression_by_cluster, index=new_data.columns), ignore_index=True)

new_data.index = Glist
new_data.to_csv("SPECIES_Mean_Gene_Expression.csv")

In [ ]:
Glist = adata.var_names
new_data = pd.DataFrame(columns=adata.obs['BREEDadd-INTESTINAL-TIME'].unique())
for gene_name in Glist[:]:
    expression = adata[:, gene_name].X
    mean_expression_by_cluster = []
    list = []
    for cluster in adata.obs['BREEDadd-INTESTINAL-TIME'].unique():
        cells_in_cluster = adata.obs['BREEDadd-INTESTINAL-TIME'] == cluster
        mean_expression = expression[cells_in_cluster].mean()
        mean_expression_by_cluster = mean_expression_by_cluster + [mean_expression]
    new_data = new_data.append(pd.Series(mean_expression_by_cluster, index=new_data.columns), ignore_index=True)

new_data.index = Glist
new_data.to_csv("BREED_Mean_Gene_Expression.csv")

# **CellTypist**

## **Global settings**

In [ ]:
import pandas as pd
import scanpy as sc
import celltypist
import time
import os
import numpy as np

In [ ]:
sc.settings.set_figure_params(dpi=100, dpi_save=300, figsize=(5, 5))

## **Ref**

In [ ]:
adata_ref = sc.read_h5ad("Full_obj_log_counts_soupx_v2.h5ad")  #PRJNA666217
sc.pp.normalize_total(adata_ref, target_sum = 1e4)
sc.pp.log1p(adata_ref)

In [ ]:
adata_ref.obs['category'].cat.categories.tolist()
# adata_ref.obs['Integrated_05'].cat.categories.tolist()

In [ ]:
sampled_cell_index = celltypist.samples.downsample_adata(adata_ref, mode = 'each', n_cells = 10000, by = 'category', return_index = True)
print(f"Number of downsampled cells for training: {len(sampled_cell_index)}")

In [ ]:
# Use `celltypist.train` to quickly train a rough CellTypist model.
# You can also set `mini_batch = True` to enable mini-batch training.
t_start = time.time()
model_fs = celltypist.train(adata_ref[sampled_cell_index], 'category', n_jobs = 10, max_iter = 5, use_SGD = True)
t_end = time.time()
print(f"Time elapsed: {t_end - t_start} seconds")

In [ ]:
gene_index = np.argpartition(np.abs(model_fs.classifier.coef_), -500, axis = 1)[:, -500:]
gene_index = np.unique(gene_index)   #>2500
print(f"Number of genes selected: {len(gene_index)}")

In [ ]:
# Add `check_expression = False` to bypass expression check with only a subset of genes.
t_start = time.time()
model = celltypist.train(adata_ref[sampled_cell_index, gene_index], 'category', check_expression = False, n_jobs = 10, max_iter = 300)
t_end = time.time()
print(f"Time elapsed: {(t_end - t_start)/60} minutes")

In [ ]:
model.write("model_human_CellLineage_ref.pkl")

## **Prediction**

In [ ]:
adata_query=sc.read_h5ad("adata_CellLineage_use.h5ad")

In [ ]:
t_start = time.time()
predictions = celltypist.annotate(adata_query, model="model_human_CellLineage_ref.pkl", majority_voting = True)
t_end = time.time()
print(f"Time elapsed: {t_end - t_start} seconds")

In [ ]:
adata_query = predictions.to_adata()

In [ ]:
sc.pl.umap(adata_query,color=['CellLineage','predicted_labels'],ncols=4,wspace=0.5,save='_CellLineage_predict.pdf')

# **Cell type proportion test**

In [ ]:
library(scProportionTest)
library(Seurat)
library(ggplot2)

In [ ]:
seurat_data <- readRDS("/disk213/xieqq/JINHUA138.sc/RDS/CellLineage.rds")

In [ ]:
prop_test <- sc_utils(seurat_data)

In [ ]:
test_result <- permutation_test(prop_test,
                                cluster_identity="CellType",
                                sample_1="du60", 
                                sample_2="du0",
                                sample_identity="BATCH") 

In [ ]:
p <- permutation_plot(test_result, FDR_threshold=0.05, log2FD_threshold=2.5, order_clusters=F)
p[["data"]][["obs_log2FD"]]
p[["data"]][["FDR"]]
p[["data"]][["clusters"]]

In [ ]:
#start
data <- as.data.frame(matrix(data=NA,nrow=0,ncol=7,dimnames=list(NULL,c("clusters","obs_log2FD","boot_CI_low","boot_CI_high","FDR","group1","group2"))))
unique(seurat_data$IT)
order <- c("small_0","small_60","small_90","small_180","small_240",
           "large_0","large_60","large_90","large_180","large_240")
for (i in 1:(length(order)-1)){
  for (j in (i+1):length(order)){
    test_result <- permutation_test(prop_test,cluster_identity="CellLineage",sample_identity="IT",
                                    sample_1=order[i],sample_2=order[j])
    p <- permutation_plot(test_result, order_clusters=F)
    newdata <- data.frame(clusters=p[["data"]][["clusters"]],
                          obs_log2FD=p[["data"]][["obs_log2FD"]],
                          boot_CI_low=p[["data"]][["boot_CI_2.5"]],
                          boot_CI_high=p[["data"]][["boot_CI_97.5"]],
                          FDR=p[["data"]][["FDR"]])
    newdata$group1 <- order[i]
    newdata$group2 <- order[j]
    data <- rbind(data,newdata)
  }
}
data$result <- "Not_Significant"
data$result[which(data$FDR<0.05&abs(data$obs_log2FD)>1.5)] <- "Significant"
write.csv(data,"permutation_test_CellLineage.csv",row.names=F)

# **RNA velocity analysis**

In [ ]:
os.chdir('/disk213/xieqq/JINHUA138.sc/velocyto')

In [ ]:
PATH='/disk213/xieqq/JINHUA138.sc/velocyto/'
ap={}
al={}
sample=['Sample_1_SI_0','Sample_2_SI_0','Sample_3_SI_0','Sample_4_SI_60','Sample_5_SI_60',
        'Sample_6_SI_60','Sample_7_SI_90','Sample_8_SI_90','Sample_9_SI_90','Sample_10_SI_180',
        'Sample_11_SI_180','Sample_12_SI_180','Sample_13_SI_240','Sample_14_SI_240','Sample_15_SI_240',
        'Sample_16_LI_0','Sample_17_LI_0','Sample_18_LI_60','Sample_19_LI_60','Sample_20_LI_90',
        'Sample_21_LI_90','Sample_22_LI_180','Sample_23_LI_180','Sample_24_LI_240','Sample_25_LI_240']

for i in sample:
    ap[f'{i}']=sc.read_loom(PATH+f'{i}'+'/'+f'{i}'+'.loom',sparse=True)
    ap[f'{i}'].var_names_make_unique
    length=len(f'{i}')+1
    ap[f'{i}'].obs.index=ap[f'{i}'].obs.index.str[length:-1]
    x=['{}','1']
    ap[f'{i}'].obs.index=ap[f'{i}'].obs.index.map('-'.join(x).format)
    y=['{}',f'{i}']
    ap[f'{i}'].obs.index=ap[f'{i}'].obs.index.map('_'.join(y).format)
    if(ap[f'{i}'].obs.shape[1]!=0):
        ap[f'{i}'].obs=ap[f'{i}'].obs.drop(columns=['Clusters','_X', '_Y'])
    al[f'{i}']=scv.utils.merge(Epithelial[Epithelial.obs['PRO1_JH'].isin([f'{i}'])], ap[f'{i}'])

In [ ]:
ldata=sc.concat(ap.values(),keys=ap.keys(),label='loom_JH')
ldata

In [ ]:
Epithelial=scv.utils.merge(Epithelial, ldata)
Epithelial

In [ ]:
Epithelial.write('Epithelial_velocyto_merge.h5ad')

In [ ]:
#scv.pp.filter_and_normalize(Epithelial, min_shared_counts=20, n_top_genes=2000)
scv.pp.moments(Epithelial)

In [ ]:
scv.tl.velocity(Epithelial)

In [ ]:
scv.tl.velocity_graph(Epithelial)

In [ ]:
scv.pl.velocity_embedding_stream(Epithelial, basis='umap', color='CellType', legend_loc='right margin')
scv.pl.velocity_embedding_stream(Epithelial, basis='umap', color='CellType', legend_loc='right margin', alpha=0, show=False, save='stream_Epithelial_right_margin.svg') 

In [ ]:
PATH='/disk213/xieqq/JINHUA138.sc/velocyto/'
sample=['Sample_1_SI_0','Sample_2_SI_0','Sample_3_SI_0','Sample_4_SI_60','Sample_5_SI_60',
        'Sample_6_SI_60','Sample_7_SI_90','Sample_8_SI_90','Sample_9_SI_90','Sample_10_SI_180',
        'Sample_11_SI_180','Sample_12_SI_180','Sample_13_SI_240','Sample_14_SI_240','Sample_15_SI_240',
        'Sample_16_LI_0','Sample_17_LI_0','Sample_18_LI_60','Sample_19_LI_60','Sample_20_LI_90',
        'Sample_21_LI_90','Sample_22_LI_180','Sample_23_LI_180','Sample_24_LI_240','Sample_25_LI_240']

for i in sample:
    al[f'{i}'].write(PATH+f'{i}'+'/'+'Epithelial_velocyto_'+f'{i}'+'.h5ad')

In [ ]:
for i in sample:
    newdata=al[f'{i}']
    scv.pp.moments(newdata)
    scv.tl.velocity(newdata)
    scv.tl.velocity_graph(newdata)
    scv.pl.velocity_embedding_grid(newdata, basis='umap', arrow_length=3, arrow_size=2, color='CellType', show=False, save='grid_Epithelial_'+f'{i}'+'.pdf')
    scv.pl.velocity_embedding_grid(newdata, basis='umap', arrow_length=3, arrow_size=2, color='CellType', alpha=0, show=False, save='grid_Epithelial_alpha0_'+f'{i}'+'.pdf') #网格线

# **Gene regulatory network inference**

## **TF**

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.pyplot import rc_context
import scanpy as sc
import loompy as lp
import pyscenic
import glob
import os

### **loom文件**

In [ ]:
os.chdir('/disk213/xieqq/JINHUA138.sc/pySCENIC')

file_in = '/disk213/xieqq/JINHUA138.sc/adata_CellLineage.h5ad'
adata = sc.read_h5ad(file_in)

file_out = 'CellLineage_scenic.loom'
newdata = adata

# create basic row and column attributes for the loom file:
row_attrs = {"Gene": np.array(newdata.var_names)}
col_attrs = {"CellID": np.array(newdata.obs_names)}
lp.create(file_out, newdata.X.transpose(), row_attrs, col_attrs)

### **TF**

In [ ]:
# transcription factors list
# f_tfs = "/disk213/xieqq/sc/ipynb/pySCENIC/allTFs_Sus_scrofa.txt" # ss
# f_tfs = "/disk213/xieqq/sc/ipynb/pySCENIC/allTFs_hg38.txt" # human
# f_tfs = "/disk213/xieqq/sc/ipynb/pySCENIC/allTFs_mm.txt"   # mouse

In [ ]:
cd /disk213/xieqq/JINHUA138.sc/pySCENIC
pyscenic grn --num_workers 5 --sparse --method grnboost2 --output CellLineage.grn.csv CellLineage_scenic.loom allTFs_hg38.txt
pyscenic ctx --num_workers 5 --output CellLineage.ctx.csv --expression_mtx_fname CellLineage_scenic.loom --mode "custom_multiprocessing" --annotations_fname motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl Epithelial.grn.csv hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather
pyscenic aucell --num_workers 3 --output CellLineage_scenic_new.loom CellLineage_scenic.loom CellLineage.ctx.csv

In [ ]:
cd /disk213/xieqq/JINHUA138.sc/pySCENIC_Epithelial
pyscenic grn --num_workers 5 --sparse --method grnboost2 --output Epithelial.grn.csv Epithelial_scenic.loom allTFs_hg38.txt
pyscenic ctx --num_workers 5 --output Epithelial.ctx.csv --expression_mtx_fname Epithelial_scenic.loom --mode "custom_multiprocessing" --annotations_fname motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl Epithelial.grn.csv hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather
pyscenic aucell --num_workers 3 --output Epithelial_scenic_new.loom Epithelial_scenic.loom Epithelial.ctx.csv

In [ ]:
arboreto_with_multiprocessing.py \
    Epithelial_scenic.loom \
    allTFs_hg38.txt \
    --method grnboost2 \
    --output Epithelial.grn.csv \
    --num_workers 10 \
    --seed 777

# **Single-cell metabolic landscape**

In [ ]:
library(scMetabolism)
library(Seurat)
library(ggplot2)
library(rsvd)
library(pheatmap)
library(dplyr)

In [ ]:
setwd("/disk213/xieqq/JINHUA138.sc/scMetabolism")

In [ ]:
pbmc = readRDS("/disk213/xieqq/JINHUA138.sc/RDS/Epithelial.rds")

In [ ]:
countexp.Seurat <- sc.metabolism.Seurat(obj = pbmc,
                                        method = "AUCell",  #ssGSEA AUCell VISION GSVA
                                        imputation = F,
                                        ncores = 2,
                                        metabolism.type = "KEGG")

# **hdWGCNA**

In [ ]:
# single-cell analysis package
library(Seurat)

# plotting and data science packages
library(tidyverse)
library(cowplot)
library(patchwork)

# co-expression network analysis packages:
library(WGCNA)
library(hdWGCNA)

# network analysis & visualization package:
library(igraph)

# using the cowplot theme for ggplot
theme_set(theme_cowplot())

# set random seed for reproducibility
set.seed(12345)

# optionally enable multithreading
enableWGCNAThreads(nThreads = 8)

In [ ]:
setwd("/disk213/xieqq/JINHUA138.sc/WGCNA")
seurat_obj <- readRDS("/disk213/xieqq/JINHUA138.sc/RDS/Epithelial.rds")

In [ ]:
seurat_obj <- SetupForWGCNA(
    seurat_obj,
    gene_select = "fraction",
    fraction = 0.05,
    wgcna_name = "tutorial"
)

In [ ]:
# metacell
seurat_obj <- MetacellsByGroups(
    seurat_obj = seurat_obj,
    group.by = c("CellType","PRO1_JH"),  
    reduction = "harmony", 
    k = 25,
    max_shared = 10, 
    ident.group = "CellType"  
)

seurat_obj <- NormalizeMetacells(seurat_obj)

In [ ]:
head(seurat_obj@misc$tutorial$wgcna_metacell_obj, 2)

In [ ]:
setwd(paste0("./",group_name))

In [ ]:
seurat_obj <- SetDatExpr(
    seurat_obj,
    group_name = group_name, 
    group.by = "CellType", 
    assay = "RNA",
    slot = "data"
)

In [ ]:
# Soft power
seurat_obj <- TestSoftPowers(
    seurat_obj,
    networkType = "signed" 
)

plot_list <- PlotSoftPowers(seurat_obj)

#wrap_plots(plot_list, ncol=2)

In [ ]:
pdf(file=paste0("Soft_power_",group_name,".pdf"), width=12, height=8)
wrap_plots(plot_list, ncol=2)
dev.off()

In [ ]:
power_table <- GetPowerTable(seurat_obj)
write.csv(power_table, paste0("Soft_power_",group_name,".csv"), row.names =F)

In [ ]:
# construct co-expression network:
seurat_obj <- ConstructNetwork(
    seurat_obj, soft_power = 12, 
    setDatExpr = FALSE,
    tom_name = group_name 
)

#PlotDendrogram(seurat_obj, main='Enterocytes hdWGCNA Dendrogram')

In [ ]:
pdf(file=paste0("hdWGCNA_Dendrogram_",group_name,".pdf"), width=12, height=8)
PlotDendrogram(seurat_obj, main=paste0(group_name," hdWGCNA Dendrogram"))
dev.off()

In [ ]:
seurat_obj@misc$tutorial$wgcna_modules %>% head
table(seurat_obj@misc$tutorial$wgcna_modules$module)
write.csv(seurat_obj@misc$tutorial$wgcna_modules,paste0("wgcna_modules_",group_name,".csv"), row.names =F)
write.csv(table(seurat_obj@misc$tutorial$wgcna_modules$module),paste0("wgcna_modules_count_",group_name,".csv"), row.names =F)

In [ ]:
TOM <- GetTOM(seurat_obj)
write.csv(TOM,paste0("TOM_",group_name,".csv"), row.names =T)

In [ ]:
# Scale
seurat_obj <- ScaleData(seurat_obj, features = VariableFeatures(seurat_obj))

seurat_obj <- ModuleEigengenes(
    seurat_obj,
    group.by.vars = "PRO1_JH"  
)

In [ ]:
# compute eigengene-based connectivity (kME):
seurat_obj <- ModuleConnectivity(
    seurat_obj,
    group.by = 'CellType', 
    group_name = group_name  
)

# module
seurat_obj <- ResetModuleNames(
    seurat_obj,
    new_name = paste0(group_name,"-M")
)

In [ ]:
pdf(file=paste0("hdWGCNA_ModulekME_",group_name,".pdf"), width=24, height=8)
PlotKMEs(seurat_obj, ncol = 4, n_hubs = 10)
dev.off()

In [ ]:
# get the module assignment table:
modules <- GetModules(seurat_obj)

# topN hub
hub_df <- GetHubGenes(seurat_obj = seurat_obj, n_hubs = 10)

saveRDS(seurat_obj, file = paste0("hdWGCNA_object_",group_name,".rds"))

In [ ]:
write.csv(modules,paste0("Module_assignment_",group_name,".csv"))
write.csv(hub_df,paste0("Module_hub_",group_name,".csv"))

In [ ]:
seurat_obj <- ModuleExprScore(
    seurat_obj,
    n_genes = 10, # topN hub genes
    method = "Seurat" # Seurat/UCell
)

In [ ]:
# group_name = "Enterocytes"
group_name = "Colonocytes"

In [ ]:
setwd(paste0("/disk213/xieqq/JINHUA138.sc/WGCNA/",group_name))
seurat_obj <- readRDS(paste0("hdWGCNA_object_",group_name,".rds"))

In [ ]:
gene_set=c("PCK1","APOE")
FeaturePlot(object = seurat_obj, features = gene_set, group.by = "ident", cols = c("blue", "red", "green"))

In [ ]:
# get hMEs from seurat object
MEs <- GetMEs(seurat_obj, harmonized=TRUE)
MEs <- MEs %>% select(sort(names(MEs)))
mods <- colnames(MEs); mods <- mods[mods != 'grey']

# mods=c("Enterocytes-M5","Enterocytes-M1","Enterocytes-M4","Enterocytes-M3","Enterocytes-M7","Enterocytes-M2","Enterocytes-M6")
# mods=c("Colonocytes-M6","Colonocytes-M4","Colonocytes-M5","Colonocytes-M2","Colonocytes-M7","Colonocytes-M1","Colonocytes-M3")

# add hMEs to Seurat meta-data:
seurat_obj@meta.data <- cbind(seurat_obj@meta.data, MEs)
write.csv(MEs,paste0("MEs_",group_name,".csv"))

In [ ]:
plot_list <- ModuleFeaturePlot(
    seurat_obj,
    #module_names = mods, #c("Enterocytes-M2","Enterocytes-M3","Enterocytes-M6","Enterocytes-M7"),
    reduction = "umap",
    features = 'hMEs', # MEs hMEs scores average
    order = TRUE # order so the points with highest hMEs are on top
)

# stitch together with patchwork
wrap_plots(plot_list, ncol=4)

pdf(file=paste0("hdWGCNA_ModulePlot_",group_name,"_hME.pdf"), width=12, height=8)
wrap_plots(plot_list, ncol=4)
dev.off()

In [ ]:
ModuleCorrelogram(seurat_obj)

pdf(file=paste0("hdWGCNA_ModuleCorrelogram_",group_name,".pdf"), width=8, height=8)
ModuleCorrelogram(seurat_obj)
dev.off()

In [ ]:
seurat_obj@meta.data$TIME <- factor(seurat_obj@meta.data$TIME, levels=c("240","180","90","60","0"))

p <- DotPlot(seurat_obj, features=mods, group.by="TIME", col.min=0, col.max=2, scale.min=0, scale.max=100) +
     RotatedAxis() + 
     scale_color_gradient(limits=c(0,2),high="#08519C", low="white")
p
ggsave(filename=paste0("hdWGCNA_Module_TIME_",group_name,".pdf"), plot=p, width=8, height=8)

In [ ]:
seurat_obj@meta.data$CellType <- factor(seurat_obj@meta.data$CellType, 
                                        levels=c('Stem','TA','Progenitor','Goblet','Tuft','EECs','BEST4enterocytes','Colonocytes','Enterocytes'),
                                        labels=c('Stem','TA','Progenitor','Goblet','Tuft','EECs','BEST4 enterocytes','Colonocytes','Enterocytes'))

p <- DotPlot(seurat_obj, features=mods, group.by="CellType", col.min=0, col.max=2, scale.min=0, scale.max=100) +
     RotatedAxis() + 
     scale_color_gradient(limits=c(0,2),high="#08519C", low="white")
p
ggsave(filename=paste0("hdWGCNA_Module_CellType_",group_name,".pdf"), plot=p, width=8, height=8)

In [ ]:
My_levels <- c('Enterocytes','Colonocytes','BEST4 enterocytes','EECs','Tuft','Goblet','Progenitor','TA','Stem')
seurat_obj$CellType <- factor(seurat_obj$CellType,levels=My_levels)
My_colors <- c('#004B23','#007200','#38B000','#52B788','#95D5B2','#D8F3DC','#34A0A4','#1A759F','#184E77')

In [ ]:
plot_list <- lapply(mods, function(x) {
  print(x)
  p <- VlnPlot(
    seurat_obj,
    features = x,
    group.by = 'CellType',
    pt.size = 0 # don't show actual data points
  )
  p <- p + geom_boxplot(width = .25, fill = "white")+xlab("") + ylab("hME") + NoLegend() + scale_fill_manual(values = My_colors)
  p
})

wrap_plots(plot_list, ncol = 4)

In [ ]:
pdf(file=paste0("hdWGCNA_violin_CellType_",group_name,".pdf"), width=12, height=8)
wrap_plots(plot_list, ncol = 4)
dev.off()

In [ ]:
p <- VlnPlot(
    seurat_obj,
    features = "Enterocytes-M2",
    group.by = "CellType",
    pt.size = 0
)
p <- p + geom_boxplot(width = .25, fill = "white")+xlab("") + ylab("hME") + NoLegend()
p

In [ ]:
FeaturePlot(seurat_obj, features=mods, ncol=4)

In [ ]:
pdf(file=paste0("hdWGCNA_umap_CellType_",group_name,".pdf"), width=16, height=8)
FeaturePlot(seurat_obj, features=mods, ncol=4)
dev.off()

In [ ]:
# using the cowplot theme for ggplot
theme_set(theme_cowplot())

# set random seed for reproducibility
set.seed(12345)  

# ModuleNetworkPlot(seurat_obj = seurat_obj, mods = "Colonocytes-M2")

In [ ]:
pdf(file=paste0("hdWGCNA_ModuleNetworkPlot_all_",group_name,".pdf"), width=8, height=8)
HubGeneNetworkPlot(
    seurat_obj,
    n_hubs = 10, 
    n_other = 5, 
    edge_prop = 0.75,
    mods = "all"
)
dev.off()

In [ ]:
cur_traits <- c('TIME','INTESTINAL')

seurat_obj <- ModuleTraitCorrelation(
  seurat_obj,
  traits = cur_traits #, 
  #group.by = 'CellType'
)
mt_cor <- GetModuleTraitCorrelation(seurat_obj)
names(mt_cor$cor)
mt_cor$cor$all_cells

write.csv(mt_cor$cor$all_cells, paste0("ModuleTraitCorrelation_",group_name,".csv"), row.names =T)
write.csv(mt_cor$pval$all_cells, paste0("ModuleTraitCorrelationPval_",group_name,".csv"), row.names =T)

In [ ]:
pdf(file=paste0("ModuleTraitCorrelation_",group_name,".pdf"), width=10, height=10)
PlotModuleTraitCorrelation(
  seurat_obj,
  label = 'pval',  
  label_symbol = 'stars',  
  text_size = 2,
  text_digits = 2,
  text_color = "black",
  high_color = "#CB1B16",
  mid_color = "white",
  low_color = "#1368AA",
  plot_max = 0.2,
  combine=TRUE
)
dev.off()

# **Pseudotime analysis - Lamian**

### **Module 1: tree variability**

In [ ]:
options(warn=-1)
suppressMessages(library(Lamian)) # load in Lamian
library(ggplot2)
library(reshape2)
library(TSCAN)
library(scattermore)
library(RColorBrewer)
suppressMessages(library(igraph))

In [ ]:
setwd("/disk213/xieqq/JINHUA138.sc/Lamian")

In [ ]:
pbmc = readRDS("/disk213/xieqq/JINHUA138.sc/RDS/Epithelial.rds")
umap <- subset(pbmc, CellType %in% c("TA","Stem","Progenitor"))

plotdir <- "./tree_variability/"

cell_type_cols=c("#34a0a4","#184e77","#1a759f")
pdf(paste0(plotdir, 'pca.pdf'), width=7, height=5)
DimPlot(umap, reduction="umap", group.by="CellType", cols=cell_type_cols, label=TRUE, label.size=3, repel=TRUE)
dev.off()

pca <- as.matrix(umap@reductions$pca@cell.embeddings)
expression <- as.matrix(umap@assays$RNA@data)
cellanno <- data.frame(cell=rownames(umap@meta.data), sample=umap@meta.data$INTESTINAL.TIME, celltype=umap@meta.data$CellType)
rownames(cellanno)=cellanno$cell

In [ ]:
### infer tree structure
res = infer_tree_structure(pca = pca,
                           expression = expression,
                           cellanno = cellanno,
                           origin.marker = c('LGR5'),
                           origin.celltype = 'Stem',
                           number.cluster = 6,
                           plotdir = plotdir,
                           xlab='Principal component 1',
                           ylab='Principal component 2')

pdf(paste0(plotdir, 'tree_structure.pdf'), width=6,height=5)
plotmclust(res, cell_point_size=0.5, x.lab='Principal component 1', y.lab = 'Principal component 2')
dev.off()

pseudotime0 <- as.data.frame(as.matrix(res[["pseudotime"]]))
pseudotime0$V2 <- names(res[["pseudotime"]])
pseudotime0 <- pseudotime0[!duplicated(pseudotime0$V2),]
pseudotime <- pseudotime0$V1
names(pseudotime) <- pseudotime0$V2
write.csv(pseudotime,paste0(plotdir, "pseudotime.csv"))

In [ ]:
### evaluate branch uncertainty
result <- evaluate_uncertainty(res, n.permute=10)
saveRDS(result, paste0(plotdir, 'result.rds'))

### **Module 3: Trajectory differential tests about gene expression**

In [ ]:
options(warn=-1)
suppressMessages(library(Lamian)) # load in Lamian
library(ggplot2)
library(reshape2)
library(TSCAN)
library(scattermore)
library(RColorBrewer)
suppressMessages(library(igraph))
library(cluster)
library(factoextra)
library(dplyr)
library(pheatmap)
library(gridExtra)

In [ ]:
setwd("/disk213/xieqq/JINHUA138.sc/Lamian")

In [ ]:
pbmc = readRDS("/disk213/xieqq/JINHUA138.sc/RDS/Epithelial.rds")
pbmc <- subset(pbmc, CellType %in% c("Colonocytes","Enterocytes"))
unique(pbmc$INTESTINAL)

plotdir <- "./trajectory_tests1/"
INTESTINAL <- "small"
umap <- subset(pbmc, INTESTINAL %in% INTESTINAL)

pdf(paste0(plotdir, INTESTINAL, '_pca.pdf'), width=6, height=5)
DimPlot(umap, reduction="pca", group.by="INTESTINAL.TIME", label=TRUE, label.size=3, repel=TRUE)
dev.off()

pca <- as.matrix(umap@reductions$pca@cell.embeddings)
expression <- as.matrix(umap@assays$RNA@data)
cellanno <- data.frame(cell=rownames(umap@meta.data), sample=umap@meta.data$INTESTINAL.TIME, 
                       celltype=sapply(strsplit(umap@meta.data$INTESTINAL.TIME, "-"), function(x) x[2]))
rownames(cellanno)=cellanno$cell

res = infer_tree_structure(pca = pca, expression = expression, cellanno = cellanno, 
                           origin.marker=c('ANPEP','FABP2'), origin.celltype=c("0"),
                           number.cluster = 5, plotdir = paste0(plotdir,INTESTINAL,"_"),
                           xlab='Principal component 1', ylab='Principal component 2')

pdf(paste0(plotdir,INTESTINAL,"_tree_structure.pdf"), width=5,height=5)
plotmclust(res,x=1,y=2,cell_point_size=0.5)
dev.off()

pseudotime <- res[["pseudotime"]]
write.csv(pseudotime,paste0(plotdir,INTESTINAL, "_pseudotime.csv"))

design = data.frame(intercept=1,group=unique(cellanno$sample))
rownames(design) <- design$group
design$group <- sapply(strsplit(design$group, "-"), function(x) x[2])

saveh5(expr=expression, pseudotime=pseudotime, cellanno=cellanno, path=paste0(plotdir,INTESTINAL,"_trajectory.h5"))

Res <- lamian_test(expr=expression, cellanno=cellanno, pseudotime=pseudotime, design=design, 
                   test.type='Time', test.method="permutation", testvar=2, permuiter=5, ncores=1)

In [ ]:
## determine the TDE genes as the genes with fdr.overall <0.05
diff_gene <- Res$statistics[Res$statistics[, 1] < 0.05,]
diffgene <- diff_gene %>% rownames()

## population fit
num.timepoint=max(pseudotime)
Res$populationFit <- getPopulationFit(Res, gene=diffgene, type='time', num.timepoint=num.timepoint)
## clustering
Res$cluster <- clusterGene(Res, gene=diffgene, type='time', k.auto=T, method="kmeans")
max(Res$cluster)
pdf(paste0(plotdir,INTESTINAL,'_cluster_mean.pdf'), width=5, height=6)
plotClusterMean(Res, cluster=Res$cluster, type='time')
dev.off()

fit <- testTDEHm_i(Res, showRowName=F, subsampleCell=F, showCluster=T, type='time')
colnames(fit) <- c(1:num.timepoint)
write.csv(fit, paste0(plotdir, INTESTINAL,"_fit.pseudotime.csv"))

outgene <- data.frame(gene=rownames(fit),order=c(1:nrow(fit)))
outcluster <- data.frame(cluster=Res$cluster)
outcluster$gene <- rownames(outcluster)
diff_gene$gene <- rownames(diff_gene)
outgene <- left_join(outgene,outcluster,by="gene")
outgene <- left_join(outgene,diff_gene,by="gene")
write.csv(outgene, paste0(plotdir, INTESTINAL,"_gene_fdr.csv"))

In [ ]:
## save png
col.expression = brewer.pal(n = 8, name = "Pastel1")[seq_len(2)]
names(col.expression) = c('Original', 'Model Fitted')
colann.fit <- data.frame(pseudotime = colnames(fit), expression = 'Model Fitted', stringsAsFactors = FALSE)
col.pseudotime = colorRampPalette(brewer.pal(n = 9, name = "YlGnBu"))(num.timepoint)
names(col.pseudotime) = colnames(fit)
clu <- Res$cluster
if (length(unique(clu)) < 8) {
  col.clu = brewer.pal(8, 'Set1')[seq_len(length(unique(clu)))]
} else {
  col.clu = colorRampPalette(brewer.pal(8, 'Set1'))[seq_len(length(unique(clu)))]
}
names(col.clu) = unique(clu)
annotation_colors = list(pseudotime = col.pseudotime,expression = col.expression,cluster = col.clu)
cpl = colorRampPalette(rev(brewer.pal(n = 7, name = "RdYlBu")))(100)
rowann = data.frame(cluster = as.character(clu),stringsAsFactors = FALSE)
rownames(rowann) = names(clu)
rowann <- rowann[rownames(fit), , drop = FALSE]
cellWidthTotal = 250;cellHeightTotal = 400

pheatmap(fit, cluster_rows = FALSE, cluster_cols = FALSE, show_rownames = FALSE, show_colnames = FALSE,
         color = cpl, annotation_col = colann.fit, annotation_row = rowann, annotation_colors = annotation_colors,
         cellwidth = cellWidthTotal / ncol(fit), cellheight = cellHeightTotal / nrow(fit), border_color = NA,
         silent = TRUE, filename = paste0(plotdir,INTESTINAL,'_pheatmap.png'))

In [ ]:
#### cluster ####
file_list <- list.files(path="/disk213/xieqq/JINHUA138.sc/Lamian/trajectory_tests", pattern="fdr.csv", full.names=TRUE)
pbmc <- readRDS("/disk213/xieqq/JINHUA138.sc/RDS/Epithelial.rds")
for (file in file_list) {
  INTESTINAL <- gsub(".*/|(\\_gene_fdr.csv$)", "", file)
  seurat_obj <- subset(pbmc, INTESTINAL %in% INTESTINAL)
  data <- read.csv(file,check.names=F,row.names=1)
  FetchData <- data.frame(cells=rownames(seurat_obj@meta.data))
  for (i in unique(data$cluster)){
    genes_list <- data$gene[which(data$cluster==i)]
    means <- data.frame(cells=colnames(seurat_obj[["RNA"]]@data[genes_list, ]),means=colMeans(seurat_obj[["RNA"]]@data[genes_list, ]))
    rownames(means) <- NULL
    FetchData <- left_join(FetchData,means,by="cells")
  }
  colnames(FetchData) <- c("cells",unique(data$cluster))
  MEs <- FetchData[,-1]
  rownames(MEs) <- rownames(FetchData)
  mods <- colnames(MEs)
  seurat_obj@meta.data <- cbind(seurat_obj@meta.data, MEs)
  FetchData$INTESTINAL.TIME = seurat_obj@meta.data$INTESTINAL.TIME
  FetchData$CellType = seurat_obj@meta.data$CellType
  write.csv(FetchData,paste0("/disk213/xieqq/JINHUA138.sc/Lamian/trajectory_tests/FetchData/Cluster_FetchData_",INTESTINAL,".csv"))
  seurat_obj@meta.data$TIME <- factor(seurat_obj@meta.data$TIME, levels=c("240","180","90","60","0")) #倒序
  p1 <- DotPlot(seurat_obj, features=mods, group.by="TIME")+RotatedAxis()+scale_color_gradient2(high="#08519C", low="#C6DBEF")
  ggsave(filename=paste0("/disk213/xieqq/JINHUA138.sc/Lamian/trajectory_tests/FetchData/Cluster_TIME_",INTESTINAL,".pdf"), plot=p1, width=8, height=8)
}